Payments raw file 

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import to_timestamp 

data = [
    Row(payment_id=40001, order_id=20001, payment_method="Credit Card", payment_status="Success", amount=87500, created_date="01-03-2025", modified_date="01-03-2025"),
    Row(payment_id=40002, order_id=20002, payment_method="UPI", payment_status="Success", amount=6500, created_date="04-03-2025", modified_date="04-03-2025"),
    Row(payment_id=40003, order_id=20003, payment_method="Net Banking", payment_status="Success", amount=13000, created_date="07-03-2025", modified_date="07-03-2025"),
    Row(payment_id=40004, order_id=20004, payment_method="UPI", payment_status="Refunded", amount=0, created_date="10-03-2025", modified_date="10-03-2025"),
    Row(payment_id=40005, order_id=20005, payment_method="Debit Card", payment_status="Success", amount=22000, created_date="13-03-2025", modified_date="13-03-2025"),
    Row(payment_id=40006, order_id=20006, payment_method="Credit Card", payment_status="Pending", amount=7000, created_date="16-03-2025", modified_date="16-03-2025"),
    Row(payment_id=40007, order_id=20007, payment_method="UPI", payment_status="Success", amount=15000, created_date="19-03-2025", modified_date="19-03-2025"),
    Row(payment_id=40008, order_id=20008, payment_method="Wallet", payment_status="Pending", amount=0, created_date="22-03-2025", modified_date="22-03-2025"),
    Row(payment_id=40009, order_id=20009, payment_method="Credit Card", payment_status="Success", amount=4400, created_date="25-03-2025", modified_date="25-03-2025"),
    Row(payment_id=40010, order_id=20010, payment_method="UPI", payment_status="Success", amount=5500, created_date="28-03-2025", modified_date="28-03-2025")
]

schema = StructType([
    StructField("payment_id", IntegerType(), False),
    StructField("order_id", IntegerType(), False),
    StructField("payment_method", StringType(), True),
    StructField("payment_status", StringType(), True),
    StructField("amount", IntegerType(), True),
    StructField("created_date", StringType(), True),
    StructField("modified_date", StringType(), True)
])

payments_raw = spark.createDataFrame(data, schema) 

payments_raw = payments_raw \
    .withColumn("created_date", to_timestamp("created_date", "dd-MM-yyyy")) \
    .withColumn("modified_date", to_timestamp("modified_date", "dd-MM-yyyy"))

display(payments_raw)

payment_id,order_id,payment_method,payment_status,amount,created_date,modified_date
40001,20001,Credit Card,Success,87500,2025-03-01T00:00:00.000Z,2025-03-01T00:00:00.000Z
40002,20002,UPI,Success,6500,2025-03-04T00:00:00.000Z,2025-03-04T00:00:00.000Z
40003,20003,Net Banking,Success,13000,2025-03-07T00:00:00.000Z,2025-03-07T00:00:00.000Z
40004,20004,UPI,Refunded,0,2025-03-10T00:00:00.000Z,2025-03-10T00:00:00.000Z
40005,20005,Debit Card,Success,22000,2025-03-13T00:00:00.000Z,2025-03-13T00:00:00.000Z
40006,20006,Credit Card,Pending,7000,2025-03-16T00:00:00.000Z,2025-03-16T00:00:00.000Z
40007,20007,UPI,Success,15000,2025-03-19T00:00:00.000Z,2025-03-19T00:00:00.000Z
40008,20008,Wallet,Pending,0,2025-03-22T00:00:00.000Z,2025-03-22T00:00:00.000Z
40009,20009,Credit Card,Success,4400,2025-03-25T00:00:00.000Z,2025-03-25T00:00:00.000Z
40010,20010,UPI,Success,5500,2025-03-28T00:00:00.000Z,2025-03-28T00:00:00.000Z


In [0]:
# spark.sql("DROP TABLE IF EXISTS catalog_project1.source1.payments_raw")
payments_raw.write.mode("append")\
                  .format("delta")\
                  .option("mergeSchema", "true")\
                  .saveAsTable("catalog_project1.source1.payments_raw")

In [0]:
spark.sql("""
SELECT * 
FROM catalog_project1.source1.payments_raw 
WHERE to_date(modified_date, 'dd-MM-yyyy') < to_date(created_date, 'dd-MM-yyyy')
""").display()

payment_id,order_id,payment_method,payment_status,amount,created_date,modified_date
